# Practical 3

## Age prediction

In this practical, we will build a machine learning model to predict a person's age based on their functional connectivity matrices. 

This is often called **brain age prediction**. The idea is that the brain changes in systematic ways across the lifespan, so a model trained on healthy brains can estimate "how old a brain looks". The difference between predicted and actual age (the *brain age gap*) has been linked to many clinical conditions.

The steps are:

1. Load the regional time series and confounds that [HALFpipe](https://github.com/HALFpipe/HALFpipe) extracted from the [fMRIPrep](https://fmriprep.org) outputs
2. **Select** which subjects to train on ← *important choice 1*
3. **Denoise** the time series by regressing out confounds ← *important choice 2*
4. Compute a **functional connectivity** matrix (Pearson correlation between every pair of regions) for each subject
5. Train a **regression model** that maps connectivity to age ← *important choice 3*
6. Estimate how well it works with **cross-validation**
7. Save the model, so that it can be evaluated on a separate dataset that you have not seen

You do not need to change any code except at the three important choices. Everything else is there so you can follow along and understand what happens.

> **Going deeper:** Boxes like this one contain more advanced notes. Feel free to skip them on a first read.

The first step is to download our training data. The [ds000030](https://www.openneuro.org/datasets/ds000030) dataset contains functional connectivity matrices for a large number of subjects, along with phenotypic data including age. 

The dataset is from the UCLA Consortium for Neuropsychiatric Phenomics. Participants are aged 21 to 50 and include healthy controls as well as people diagnosed with schizophrenia, bipolar disorder, and ADHD. We use the resting state scans, which are about five minutes long (152 volumes).

The data is managed with [DataLad](https://www.datalad.org), which keeps track of large files with git-annex. `datalad get` downloads the actual file contents.

In [ ]:
%%bash
datalad get --jobs 4 ds000030-halfpipe

In [ ]:
%%bash
datalad get atlas-Schaefer2018Combined_dseg.tsv

We also need to make sure we have the necessary Python packages installed. We will use `nilearn` and `scikit-learn` for this practical. We are using the `subprocess` syntax to make sure we install them to the correct place.

In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "nilearn==0.13.1", "scikit-learn==1.9.1"],
    check=False,
)

For each subject, we have two tab-separated files:

- `*_timeseries.tsv` contains the mean BOLD signal in each brain region of an atlas. There is one row per time point (volume) and one column per region.
- `*_desc-confounds_timeseries.tsv` contains the confound time series computed by fMRIPrep. There is also one row per time point, and one column per confound (see below).

The `TimeSeries` class below is a small helper that keeps the paths to both files together and loads them on demand.

In [ ]:
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from numpy import typing as npt


@dataclass
class TimeSeries:
    """
    Represents a time series.

    Attributes:
        path (Path): The path to the ".tsv" file containing the time series.
        metadata (dict[str, Any]): Additional metadata associated with the connectivity matrix.
    """

    path: Path
    confounds_path: Path

    def __post_init__(self) -> None:
        if not self.path.is_file() and self.confounds_path.is_file():
            raise ValueError("Files not found")
    
    @property
    def subject(self) -> str:
        return self.path.name.split("_")[0]
    
    @property
    def values(self) -> npt.NDArray[np.float64]:
        """
        Time series values from the file.

        Returns:
            ndarray: The time series as a NumPy array.
        """
        return np.loadtxt(self.path, delimiter="\t")
    
    @property
    def confounds(self) -> pd.DataFrame:
        """
        Associated confound time series.

        Returns:
            ndarray: The confounds time series as a Pandas data frame.
        """
        return pd.read_table(self.confounds_path)

Next, we find all resting state time series files in the dataset and pair each one with its confounds file.

In [ ]:
time_series = [
    TimeSeries(
        path=path,
        confounds_path=path.parent / path.name.replace("timeseries", "desc-confounds_timeseries"),
    )
    for path in Path("ds000030-halfpipe").rglob("*_task-rest_timeseries.tsv")
]

Let's look at the data of the first subject. In this plot, each row is a brain region and the x-axis is time. We z-score each region so that they are all on the same scale. The regions come from the [Schaefer 2018](https://doi.org/10.1093/cercor/bhx179) atlas with 400 cortical regions, plus subcortical and cerebellar regions. We sort them by the brain network they belong to, which is shown by the colored bar on the left. This is similar to the "carpet plots" used for fMRI quality control. Vertical stripes that span many regions are often caused by head motion or other artefacts, which is why we need denoising.

In [ ]:
from typing import Any

import scipy.stats
from matplotlib import pyplot as plt
from matplotlib.patches import Rectangle

atlas = pd.read_table("atlas-Schaefer2018Combined_dseg.tsv")

# Sort the regions so that regions of the same network are next to each other
atlas["network"] = pd.Categorical(atlas.network, categories=atlas.network.unique())
atlas = atlas.sort_values("network", kind="stable")
atlas["position"] = np.arange(len(atlas))


def add_network_labels(axes):
    """Draw a colored bar and a label for each network on the left of the plot"""
    for network, regions in atlas.groupby("network", observed=True):
        start = regions.position.min() - 0.5
        end = regions.position.max() + 0.5
        bar = Rectangle(
            (-0.03, start),
            width=0.03,
            height=end - start,
            color=regions.color.iloc[0],
            transform=axes.get_yaxis_transform(),
            clip_on=False,
        )
        axes.add_patch(bar)
        axes.axhline(end, color="black", linewidth=0.5)  # Line between networks

    ticks = atlas.groupby("network", observed=True).position.mean()
    axes.set_yticks(ticks, ticks.index)
    axes.tick_params(axis="y", length=0, pad=16)


repetition_time = 2  # Seconds between two volumes

# Get an example time series and reorder its regions according to the atlas
values: Any = time_series[0].values[:, atlas.index]
values = scipy.stats.zscore(values, axis=0, nan_policy="omit")
duration = len(values) * repetition_time

figure, axes = plt.subplots(figsize=(8, 5))
image = axes.imshow(
    values.transpose(),
    cmap="gray",
    vmin=-3,
    vmax=3,
    aspect="auto",
    interpolation="nearest",
    extent=(0, duration, len(atlas) - 0.5, -0.5),
)
axes.set_title(f"Time series of {time_series[0].subject}")
axes.set_xlabel("Time (seconds)")
add_network_labels(axes)

Now we load the age of each participant from `participants.tsv`. We use `.loc` to put the rows in the same order as our list of time series, so that the *i*-th age belongs to the *i*-th subject. By convention, the variable we want to predict (the *target*) is called `y`.

We can use `data_frame.describe()` to get a quick overview of the age distribution in our training data.

> **Going deeper:** Ages in the training data range from 21 to 50, with a mean of about 33 years. A model cannot be expected to extrapolate to ages it has never seen.

In [ ]:
data_frame = (
    pd.read_table("ds000030-halfpipe/participants.tsv")
    .set_index("participant_id")
    .loc[[time_series.subject for time_series in time_series]]
)
y = pd.Series(data_frame.age)
data_frame.describe()

### Important choice: subjects

More data usually means a better model, but only if the data is relevant and of good quality. `participants.tsv` has some information that you can use to choose which subjects to train on:

| Option | Column in `participants.tsv` | Subjects | Why you might choose it | What it costs |
|---|---|---|---|---|
| Everyone | — | 194 | The largest sample, which helps the model learn more reliable patterns | The sample includes people with schizophrenia, bipolar disorder, and ADHD. These conditions also change connectivity, and the patient groups are slightly older on average. The model may partly learn "patient-like connectivity" as a shortcut for "older" |
| Controls only | `diagnosis` is `CONTROL` | 91 | Brain age models are usually trained on healthy people, so that the brain age gap can later be interpreted as a deviation from healthy ageing | Less than half of the subjects |
| No ghosting | `ghost_NoGhost` is `No_ghost` | 163 | Some scans have *ghosting*, an artefact where a shifted copy of the image appears, and were flagged during quality control by the [dataset curators](https://f1000research.com/articles/6-1262/v2) | Fewer subjects. Ghosting is more common in some patient groups, so this also changes the sample composition |

You can combine the options, for example to use only controls without ghosting. Remove the `#` in front of an option in the cell below to activate it.

**Watch out:** the cell below removes subjects from the list, so if you change your choice, re-run the notebook from the top.

> **Going deeper:** Every exclusion is a trade-off between bias and variance. A smaller but cleaner sample can generalize better than a large noisy one, or worse, depending on how much the excluded subjects differ from the test data. Exclusions can also change the age distribution, so check `y.describe()` after you have made your choice.

In [ ]:
# Controls only
# data_frame = data_frame.loc[data_frame.diagnosis == "CONTROL"]

# No ghosting
# data_frame = data_frame.loc[data_frame.ghost_NoGhost == "No_ghost"]

# Keep only the selected subjects
time_series = [t for t in time_series if t.subject in data_frame.index]
y = pd.Series(data_frame.age)
print(f"Using {len(time_series)} subjects")

### Important choice: denoising

The BOLD signal does not only reflect neural activity. It also contains noise from head motion, breathing, heart beat, and scanner drift. These noise sources are *shared* across many regions, so they create spurious correlations and inflate functional connectivity. Worse, some of them are related to age: for example, how much people move in the scanner differs between age groups. If we do not remove them, our model may learn to predict age from motion instead of from the brain.

fMRIPrep estimates many *confound time series* for every scan. We remove them with linear regression: for every region, we fit a model that predicts the region's signal from the confounds, and keep only the residuals (the part that the confounds cannot explain).

These are the columns available in the confounds files:

| Column(s) | What it measures | Notes |
|---|---|---|
| `trans_x`, `trans_y`, `trans_z`, `rot_x`, `rot_y`, `rot_z` | Estimated head motion: translation (mm) and rotation (radians) along each axis | The "6 motion parameters". The most basic denoising strategy |
| `*_derivative1` | Change from the previous time point (temporal derivative) | Exists for motion parameters and for `white_matter`, `csf` and `global_signal`. The first row is empty (`n/a`), but that's okay because it's filled in by the imputer in the code below |
| `*_power2`, `*_derivative1_power2` | Squares of the above | Capture non-linear effects of motion. 6 parameters + derivatives + squares = the "24-parameter" model ([Friston et al., 1996](https://doi.org/10.1002/mrm.1910350312)) |
| `framewise_displacement` | A single number per time point that summarises how much the head moved since the previous one, in mm ([Power et al., 2012](https://doi.org/10.1016/j.neuroimage.2011.10.018)) | Mostly used for quality control and to decide which volumes or subjects to exclude, rather than as a regressor |
| `rmsd` | Root mean square displacement, another summary of motion (Jenkinson, 1999) | Similar use as `framewise_displacement` |
| `dvars`, `std_dvars` | How much the whole-brain signal changes from one time point to the next | A quality control measure. Spikes indicate artefacts. Rarely used as a regressor |
| `white_matter`, `csf` | Mean signal in white matter and in cerebrospinal fluid | There should be little neural signal there, so these mostly capture physiological noise. Commonly used together with the motion parameters |
| `csf_wm` | Mean signal in white matter and cerebrospinal fluid combined | An alternative to using `white_matter` and `csf` separately |
| `global_signal` | Mean signal across the whole brain | *Global signal regression* (GSR) removes motion and breathing artefacts very effectively, but it also shifts all correlations so that some become negative, and it may remove real neural signal. It is a much-debated choice |
| `a_comp_cor_00`, `a_comp_cor_01`, ... | *Anatomical CompCor* ([Behzadi et al., 2007](https://doi.org/10.1016/j.neuroimage.2007.04.042)): principal components of the signal in white matter and CSF combined | Captures noise better than the mean signal alone. Components are sorted by how much variance they explain, so `_00` is the most important. Using the first 5 is common |
| `c_comp_cor_*`, `w_comp_cor_*` | The same as `a_comp_cor`, but only for CSF or only for white matter | An alternative to `a_comp_cor` |
| `t_comp_cor_*`, `tcompcor` | *Temporal CompCor*: principal components of the voxels with the highest variance over time | High-variance voxels tend to be near vessels and edges, so these capture physiological noise |
| `edge_comp_*` | Principal components of the signal in a thin band around the edge of the brain ([Patriat et al., 2017](https://doi.org/10.1016/j.neuroimage.2016.08.051)) | The edge of the brain is very sensitive to motion |
| `cosine00`, `cosine01`, `cosine02` | Slow cosine waves that act as a high-pass filter (removing fluctuations slower than 1/128 Hz) | Removes scanner drift. Longer scans have more of these columns. fMRIPrep computes CompCor on high-pass filtered data, so you should include these whenever you use any `*comp_cor*` columns |

There is no single best strategy. Benchmarking studies such as [Ciric et al., 2017](https://doi.org/10.1016/j.neuroimage.2017.03.020) and [Parkes et al., 2018](https://doi.org/10.1016/j.neuroimage.2017.12.073) found that different strategies are better at removing different kinds of artefacts. Try a few and see what happens to the cross-validated performance below!

**Watch out:** the number of CompCor and edge components differs between subjects, because fMRIPrep keeps as many components as it needs to explain 50% of the variance. Your columns must exist for *every* subject, both in this dataset and in the dataset we will use for evaluation, otherwise you will get a `KeyError`. You can use at most the first 22 `a_comp_cor`, 24 `w_comp_cor`, 3 `c_comp_cor`, 1 `t_comp_cor`, 24 `edge_comp`, and 3 `cosine` columns. In the cell below, `f"a_comp_cor_{i:02d}" for i in range(5)` creates the names of the first five aCompCor columns, from `a_comp_cor_00` to `a_comp_cor_04`, so that you do not have to type them by hand.

> **Going deeper:** Each regressor costs one degree of freedom. Our scans only have 152 time points, so 36P plus cosines already uses up about a quarter of them. Removing too much can also remove real signal and add variance to the connectivity estimates. Also consider that the confounds themselves can carry information about age. A model that performs *better* with less denoising may simply be exploiting motion. [nilearn's `load_confounds_strategy`](https://nilearn.github.io/stable/modules/generated/nilearn.interfaces.fmriprep.load_confounds_strategy.html) implements many of these strategies, if you want to see how they are defined.

Remove the `#` in front of the lines you want to use in the list below, or add your own.

In [ ]:
# Remove the "#" at the start of a line to use those confounds.
# The "*" puts multiple elements into this list.
confounds_columns = [
    ### Motion parameters
    # *(f"{param}_{axis}" for param in ["trans", "rot"] for axis in ["x", "y", "z"]),
    ### Derivatives of motion parameters
    # *(f"{param}_{axis}_derivative1" for param in ["trans", "rot"] for axis in ["x", "y", "z"]),
    ### Squares of motion parameters
    # *(f"{param}_{axis}_power2" for param in ["trans", "rot"] for axis in ["x", "y", "z"]),
    ### Squares of derivatives of motion
    # *(f"{param}_{axis}_derivative1_power2" for param in ["trans", "rot"] for axis in ["x", "y", "z"]),
    ### White matter
    # "white_matter",
    ### CSF (cerebrospinal fluid)
    # "csf",
    ### Global signal (average signal across the whole brain)
    # "global_signal",
    ### Top five aCompCor components
    # *(f"a_comp_cor_{i:02d}" for i in range(5)),
    ### Top tCompCor component (only one is available for all subjects)
    # "t_comp_cor_00",
    ### Cosine regressors for high-pass filtering (use them together with CompCor)
    # *(f"cosine{i:02d}" for i in range(3)),
]

print(f"Using {len(confounds_columns)} confounds: {confounds_columns}")

Now we compute one connectivity matrix per subject. For each subject, the function below:

1. Fills in missing values in the confounds (`SimpleImputer`), standardizes them (`StandardScaler`), and regresses them out of every region's time series (`LinearRegression`). We keep the residuals.
2. Computes the Pearson correlation between every pair of regions. The result is a square matrix with one row and one column per region.

A connectivity matrix is symmetric (the correlation of A with B is the same as B with A), and the diagonal is always 1. So we only keep the values below the diagonal and flatten them into one long vector per subject with `sym_matrix_to_vec`. Stacking these vectors gives us `X`, a table with one row per subject and one column per pair of regions. These are the *features* for our model.

> **Going deeper:** With 434 regions, every subject has 434 × 433 / 2 = 93,961 features, but we only have 194 subjects. When there are far more features than samples, a model can fit the training data perfectly and still generalize badly (*overfitting*). This is why regularization and dimensionality reduction are so important in the next steps. Note also that denoising is done *within* each subject, so it does not leak information between subjects during cross-validation.

In [ ]:
from nilearn.connectome import sym_matrix_to_vec
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm


def calculate_correlation_matrix(time_series: TimeSeries) -> npt.NDArray[np.float64]:
    confounds = time_series.confounds[confounds_columns].to_numpy()
    values = np.nan_to_num(time_series.values)

    if confounds_columns:
        model = make_pipeline(
            SimpleImputer(strategy="constant"), StandardScaler(), LinearRegression()
        )

        predicted_values = model.fit(confounds, values).predict(confounds)
        values = values - predicted_values

    return pd.DataFrame(values).corr().to_numpy()


correlation_matrices = np.asarray(
    [calculate_correlation_matrix(time_series) for time_series in tqdm(time_series)]
)

X = sym_matrix_to_vec(correlation_matrices, discard_diagonal=True)

Let's look at the connectivity matrix of the first subject. The regions are sorted by network, like in the time series plot above. The colorbar shows which colors stand for positive and negative correlations.

Things to look for:

- With good denoising, you should see squares of strong positive correlation along the diagonal. Regions in the same network are more strongly correlated with each other than with other regions.
- Without denoising, almost all correlations are strongly positive. Noise that is shared across the whole brain makes every pair of regions look correlated. Try different confounds and compare!
- White lines are regions without signal, for example because of signal dropout near the sinuses (`Limbic` regions in the orbitofrontal cortex and temporal pole).

In [ ]:
# Get an example correlation matrix and reorder it according to the atlas
matrix = correlation_matrices[0]
matrix = matrix[atlas.index, :][:, atlas.index]

figure, axes = plt.subplots(figsize=(8, 7))
image = axes.imshow(matrix, cmap="managua_r", vmin=-1, vmax=1, interpolation="nearest")
figure.colorbar(image, label="Pearson correlation", ax=axes)
axes.set_title(f"Functional connectivity of {time_series[0].subject}")
add_network_labels(axes)

# Add vertical lines between networks
for end in atlas.groupby("network", observed=True).position.max() + 0.5:
    axes.axvline(end, color="black", linewidth=0.5)
axes.set_xticks([])

### Important choice: estimator

The estimator is the machine learning model that learns the relationship between connectivity (`X`) and age (`y`). Each estimator has *hyperparameters*, which are settings that you choose rather than learn from the data. Here are three good ideas to try:

| Estimator | How it works | Key hyperparameters | Notes |
|---|---|---|---|
| [`Ridge(alpha=1.0)`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html) | Linear regression with a penalty that shrinks all weights towards zero | `alpha`: larger means more shrinkage. Try values from `0.01` to `10000` | A robust default for neuroimaging data with many correlated features. [`RidgeCV`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.RidgeCV.html) picks `alpha` for you, for example `RidgeCV(alphas=np.logspace(-2, 4, 20))` |
| [`SVR(kernel="linear", C=1.0)`](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVR.html) | Support vector regression: ignores errors smaller than `epsilon` and penalizes larger ones linearly | `C`: smaller means more regularization. `epsilon`: the error (in years) that is ignored | Popular in the brain age literature. Less sensitive to outliers than `Ridge`. With `kernel="rbf"` it can also learn non-linear relationships |
| [`RandomForestRegressor(n_estimators=500)`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html) | Averages the predictions of many decision trees, each trained on a random subset of the data | `n_estimators`: the number of trees. `max_depth`: how complex each tree can be | Non-linear and needs little tuning. Slower to train |

You need to import the estimator first, for example `from sklearn.svm import SVR` or `from sklearn.ensemble import RandomForestRegressor`. To see what performance you get without learning anything, try [`DummyRegressor()`](https://scikit-learn.org/stable/modules/generated/sklearn.dummy.DummyRegressor.html) from `sklearn.dummy`, which always predicts the mean age. scikit-learn has [many more estimators](https://scikit-learn.org/stable/supervised_learning.html) if you want to explore. You can also change other steps of the pipeline in the next cell, for example the number of PCA components (`n_components`).

> **Going deeper:** Linear models with strong regularization are hard to beat in neuroimaging when the sample size is small ([Schulz et al., 2020](https://doi.org/10.1038/s41467-020-18037-z)). If you tune hyperparameters by hand while looking at the cross-validation scores below, those scores become optimistic, because you have used them to make your choice. Estimators like `RidgeCV` avoid this by tuning inside each training fold (this is called *nested cross-validation*). You can do the same for any estimator by wrapping it in [`GridSearchCV`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html).

In [ ]:
from sklearn.linear_model import Ridge

estimator = Ridge(alpha=1.0)

The estimator is the last step of a scikit-learn `Pipeline`. A pipeline chains several steps, so that each one feeds into the next:

1. `SimpleImputer` replaces missing values (for example from regions without signal) with the median of that feature.
2. `VarianceThreshold` removes features that are (almost) constant across subjects, because they carry no information.
3. `StandardScaler` rescales every feature to mean 0 and standard deviation 1, so that no feature dominates just because of its scale.
4. `PCA` (principal component analysis) compresses the ~94,000 features into 50 components that capture as much of the variation between subjects as possible.
5. The estimator predicts age from these 50 components.

> **Going deeper:** Putting all steps into one pipeline is not just convenient. During cross-validation, every step is fitted *only on the training subjects* of each fold. If we instead ran the scaler or PCA on all subjects first, information from the test subjects would leak into training, and our performance estimates would be too optimistic (*data leakage*).

In [ ]:
from sklearn.decomposition import PCA
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline

model = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
        ("variance_threshold", VarianceThreshold(threshold=1e-6)),
        ("scaler", StandardScaler()),
        (
            "pca",
            PCA(
                n_components=50,
                svd_solver="randomized",
                random_state=0,
            ),
        ),
        ("estimator", estimator),
    ]
)

To estimate how well our model works on *new* subjects, we use **5-fold cross-validation**. The subjects are split into five groups. We train the model on four of them and test it on the fifth, and repeat this five times so that every subject is tested once. We report two scores for each fold:

- **Mean absolute error (MAE)**: the average difference between predicted and actual age, in years. Lower is better. scikit-learn always maximizes scores, so it reports the *negative* MAE. Just ignore the minus sign.
- **R²** (coefficient of determination): the fraction of the variance in age that the model explains. 1 is perfect, 0 is no better than always predicting the mean, and it can even be negative if the model is worse than that.

For comparison, always predicting the mean age gives an MAE of about 8.3 years when using all subjects. You can check this yourself by setting the estimator to `DummyRegressor()`.

> **Going deeper:** The scores vary quite a bit between folds. With fewer than 200 subjects, differences of a few tenths of a year between two pipelines are likely to be noise ([Varoquaux, 2018](https://doi.org/10.1016/j.neuroimage.2017.06.061)). Also, the model will be evaluated on a different dataset, acquired at a different site with a different scanner and protocol. Cross-validation within one dataset tells you nothing about how well the model handles such differences. Choices that look slightly worse here may generalize better.

In [ ]:
cross_validate(
    model, X, y, scoring=("neg_mean_absolute_error", "r2"), verbose=3, n_jobs=4
)

Once you are happy with your choices, we train the final model on *all* subjects and save it to `checkpoint.joblib`, together with the list of confounds you used. We need the confounds so that the evaluation can denoise the test data in exactly the same way. Submit this file to https://forms.gle/DEJQo6u1HnTu1iVv5. You can upload multiple checkpoints if you want to try different pipelines. The evaluation will use your last submission.

In [ ]:
import joblib

model.fit(X, y)

joblib.dump(
    {
        "model": model,
        "confounds_columns": confounds_columns,
    },
    "checkpoint.joblib",
)